In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
# Crear widgets
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsfinalproject")

In [0]:
# Obtener valores de widgets
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

# Obtener ruta de races.csv
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/races.csv"

In [0]:
# Definir esquema para el DataFrame de races
races_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                  StructField("year", IntegerType(), True),
                                  StructField("round", IntegerType(), True),
                                  StructField("circuitId", IntegerType(), True),
                                  StructField("name", StringType(), True),
                                  StructField("date", DateType(), True),
                                  StructField("time", StringType(), True),
                                  StructField("url", StringType(), True) 
])

In [0]:
# Crear Dataframe races_df con el esquema anterior.
races_df = spark.read \
            .option("header", True) \
            .schema(races_schema) \
            .csv(ruta)

In [0]:
# Añadir columna ingestion_date
races_with_timestamp_df = races_df.withColumn("ingestion_date", current_timestamp())

In [0]:
# Renombrar columnas.
races_selected_df = races_with_timestamp_df.select(col('raceId').alias('race_id'), 
                                                   col('year').alias('race_year'), 
                                                   col('round'), col('circuitId').alias('circuit_id'),
                                                   col('name'), col('ingestion_date'))

In [0]:
# Ingestar data en la tabla races.
races_selected_df.write.mode('overwrite').partitionBy('race_year').saveAsTable(f'{catalogo}.{esquema}.races')